# Step 3 — Systems: B0 / P2 / P3 (all SNRs)

| ID | System |
|----|--------|
| **B0** | ECAPA cosine only (noisy test) |
| **P2** | Wave-U-Net + **SNR-gated** emb fusion |
| **P3** | Always-enhance emb |

Enrollment stays **clean**. Loops **clean + SNR 15/10/5/0** on **dev**.

MUSAN path: `D:/downloads/musan/musan/noise`  
Wave-U-Net: `app/server/checkpoints/waveunet_finetuned_v4_best.pt`

`SMOKE = False` = full trials (slow). Set `True` for a 150-trial debug.

**Jupyter kernel must be** `app/server/.venv` (has `denoisers` for Wave-U-Net).


In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_LA,
    DEFAULT_SASV,
    DEFAULT_SNRS_DB,
    RUNS_DIR,
    build_speaker_models,
    cosine,
    eers_from_preds,
    embed_waveform,
    ensure_dirs,
    ensure_sasv_on_path,
    ensure_server_on_path,
    fuse_embeddings,
    load_app_ecapa,
    load_enhancer,
    load_waveform,
    maybe_noise_waveform,
    patch_speechbrain_windows_lazy_import,
    read_trials,
    resolve_audio_path,
    resolve_noise_bank,
    save_json,
    snr_gate_weight,
    snr_tag,
    trial_key_counts,
    write_score_csv,
)

ensure_dirs()
ensure_server_on_path()
patch_speechbrain_windows_lazy_import()

SMOKE = False
SPLIT = "dev"
MAX_TRIALS = 150 if SMOKE else 0  # 0 = all trials
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 20260924
RNG = np.random.default_rng(SEED)

NOISE_ROOT = Path(r"D:/downloads/musan/musan/noise")
noise_bank, noise_used = resolve_noise_bank(NOISE_ROOT)

print("device", DEVICE, "split", SPLIT, "trials_cap", MAX_TRIALS)
print("SNR grid", [snr_tag(s) for s in DEFAULT_SNRS_DB])
print("MUSAN", noise_used, "clips", len(noise_bank))


[noise] using 62 clips from D:\downloads\musan\musan\noise
device cuda:0 split dev trials_cap 0
SNR grid ['clean', 'snr15db', 'snr10db', 'snr5db', 'snr0db']
MUSAN D:\downloads\musan\musan\noise clips 62


### Load ECAPA + Wave-U-Net


In [2]:
sasv = ensure_sasv_on_path(DEFAULT_SASV)
trials = read_trials(sasv, SPLIT, max_trials=MAX_TRIALS)
print(trial_key_counts(trials))

classifier = load_app_ecapa(device=DEVICE)
enhancer = load_enhancer(device=DEVICE)
assert enhancer is not None, (
    "Wave-U-Net failed. Select Jupyter kernel: app/server/.venv "
    "(denoisers is installed there). Or: python -m pip install denoisers"
)

spk_models = build_speaker_models(classifier, DEFAULT_LA, SPLIT, DEVICE)
print("speakers", len(spk_models))


{'target': 1484, 'nontarget': 5768, 'spoof': 22296, 'total': 29548}
[WARN] webrtc_noise_gain not installed — WebRTC disabled.
[noise_gated_lib] enhancer ok: WaveUNetEnhancer ckpt=D:\speaker-verification-system\app\server\checkpoints\waveunet_finetuned_v4_best.pt


Enrol dev:   0%|          | 0/10 [00:00<?, ?it/s]

speakers 10


### Score all SNRs (B0 / P2 / P3)


In [3]:
@torch.inference_mode()
def score_systems(trials, snr_db):
    rows_b0, rows_p2, rows_p3 = [], [], []
    preds = {"B0": [], "P2": [], "P3": []}
    keys = []

    for trial in tqdm(trials, desc=f"score:{snr_tag(snr_db)}"):
        path = resolve_audio_path(DEFAULT_LA, SPLIT, trial.test_utt)
        clean = load_waveform(path)
        test_wave = maybe_noise_waveform(
            clean, snr_db=snr_db, noise_bank=noise_bank, rng=RNG
        )

        emb_raw = embed_waveform(classifier, test_wave, DEVICE)
        enh_wave = enhancer.process(test_wave.cpu())
        if not torch.is_tensor(enh_wave):
            enh_wave = torch.as_tensor(enh_wave, dtype=torch.float32)
        emb_enh = embed_waveform(classifier, enh_wave.float(), DEVICE)

        spk = spk_models[trial.speaker_id]
        s_b0 = float(cosine(spk, emb_raw))
        s_p2 = float(cosine(spk, fuse_embeddings(emb_raw, emb_enh, mode="gated", snr_db=snr_db)))
        s_p3 = float(cosine(spk, fuse_embeddings(emb_raw, emb_enh, mode="always_enhance", snr_db=snr_db)))

        keys.append(trial.key)
        preds["B0"].append(s_b0)
        preds["P2"].append(s_p2)
        preds["P3"].append(s_p3)
        base = {
            "speaker_id": trial.speaker_id,
            "test_utt": trial.test_utt,
            "key": trial.key,
            "snr": snr_tag(snr_db),
        }
        rows_b0.append({**base, "score": s_b0})
        rows_p2.append({**base, "score": s_p2, "w_enhanced": snr_gate_weight(snr_db)})
        rows_p3.append({**base, "score": s_p3})

    return rows_b0, rows_p2, rows_p3, preds, keys

all_summaries = {}
for snr_db in (10, 5, 0):  # skip clean + snr15 — already done
    rows_b0, rows_p2, rows_p3, preds, keys = score_systems(trials, snr_db)
    tag = snr_tag(snr_db)
    out = RUNS_DIR / f"step3_{SPLIT}_{tag}"
    out.mkdir(parents=True, exist_ok=True)

    write_score_csv(out / "B0_scores.csv", rows_b0, ["speaker_id", "test_utt", "key", "snr", "score"])
    write_score_csv(out / "P2_scores.csv", rows_p2, ["speaker_id", "test_utt", "key", "snr", "score", "w_enhanced"])
    write_score_csv(out / "P3_scores.csv", rows_p3, ["speaker_id", "test_utt", "key", "snr", "score"])

    summaries = {}
    for name in ("B0", "P2", "P3"):
        summaries[name] = {
            "system": name,
            "split": SPLIT,
            "snr": tag,
            "max_trials": MAX_TRIALS,
            **eers_from_preds(preds[name], keys, DEFAULT_SASV),
        }
    save_json(out / "metrics.json", summaries)
    all_summaries[tag] = summaries
    print(tag, {k: round(v["sasv_eer_percent"], 3) for k, v in summaries.items()})

save_json(RUNS_DIR / f"step3_{SPLIT}_all_snrs.json", all_summaries)
all_summaries


score:snr10db:   0%|          | 0/29548 [00:00<?, ?it/s]

snr10db {'B0': 17.136, 'P2': 18.055, 'P3': 22.417}


score:snr5db:   0%|          | 0/29548 [00:00<?, ?it/s]

snr5db {'B0': 18.012, 'P2': 20.081, 'P3': 22.035}


score:snr0db:   0%|          | 0/29548 [00:00<?, ?it/s]

snr0db {'B0': 19.548, 'P2': 22.035, 'P3': 22.488}


{'snr10db': {'B0': {'system': 'B0',
   'split': 'dev',
   'snr': 'snr10db',
   'max_trials': 0,
   'sasv_eer': 0.17135832383159844,
   'sv_eer': 0.022911051212938075,
   'spf_eer': 0.20350404312668469,
   'sasv_eer_percent': 17.135832383159844,
   'sv_eer_percent': 2.2911051212938074,
   'spf_eer_percent': 20.35040431266847},
  'P2': {'system': 'P2',
   'split': 'dev',
   'snr': 'snr10db',
   'max_trials': 0,
   'sasv_eer': 0.18055159635101714,
   'sv_eer': 0.024618585298528665,
   'spf_eer': 0.21080014352309215,
   'sasv_eer_percent': 18.055159635101713,
   'sv_eer_percent': 2.4618585298528664,
   'spf_eer_percent': 21.080014352309213},
  'P3': {'system': 'P3',
   'split': 'dev',
   'snr': 'snr10db',
   'max_trials': 0,
   'sasv_eer': 0.22416619156174514,
   'sv_eer': 0.07681940700808619,
   'spf_eer': 0.2555615357023306,
   'sasv_eer_percent': 22.416619156174512,
   'sv_eer_percent': 7.681940700808619,
   'spf_eer_percent': 25.556153570233057}},
 'snr5db': {'B0': {'system': 'B0',
   

### Done when
- `runs/step3_dev_{clean,snr15db,...}/metrics.json` exist for every SNR
- Console shows MUSAN clip count > 0

Next → **04_matrix_tables_and_claim.ipynb**
